In [12]:
#langchain连接neo4j
from langchain_neo4j import Neo4jGraph
from configuration.config import *

graph = Neo4jGraph(
    url=NEO4J_CONFIG["uri"],
    username=NEO4J_CONFIG["auth"][0],
    password=NEO4J_CONFIG["auth"][1],
)

print("graph created")
print(graph.schema)


graph created
Node properties:
Category1 {id: INTEGER, name: STRING}
Category2 {id: INTEGER, name: STRING}
Category3 {id: INTEGER, name: STRING}
BaseAttrName {id: INTEGER, name: STRING}
BaseAttrValue {id: INTEGER, name: STRING}
SPU {id: INTEGER, name: STRING}
SKU {id: INTEGER, name: STRING}
Trademark {id: INTEGER, name: STRING}
SaleAttrName {id: INTEGER, name: STRING}
SaleAttrValue {id: INTEGER, name: STRING}
Tag {id: STRING, name: STRING}
Relationship properties:

The relationships:
(:Category1)-[:Have]->(:BaseAttrName)
(:Category2)-[:Belong]->(:Category1)
(:Category2)-[:Have]->(:BaseAttrName)
(:Category3)-[:Have]->(:BaseAttrName)
(:Category3)-[:Belong]->(:Category2)
(:BaseAttrName)-[:Have]->(:BaseAttrValue)
(:SPU)-[:Have]->(:Tag)
(:SPU)-[:Have]->(:SaleAttrName)
(:SPU)-[:Belong]->(:Trademark)
(:SPU)-[:Belong]->(:Category3)
(:SKU)-[:Have]->(:BaseAttrValue)
(:SKU)-[:Have]->(:SaleAttrValue)
(:SKU)-[:Belong]->(:SPU)
(:SaleAttrName)-[:Have]->(:SaleAttrValue)


In [21]:
#定义大模型
from langchain_deepseek import ChatDeepSeek

llm = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0,
    api_key=DEEPSEEK_API_KEY
)


In [22]:
#定义chain
from langchain_neo4j import GraphCypherQAChain

# 定义chain
chain = GraphCypherQAChain.from_llm(
    graph=graph,
    llm=llm,
    verbose=True,
    allow_dangerous_requests=True
)
result = chain.invoke({"query": "华为有哪些产品？"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (t:Trademark {name: '华为'})<-[:Belong]-(spu:SPU)
RETURN spu.name AS product_name

Full Context:
[{'product_name': '华为智慧屏 4K全面屏智能电视机'}, {'product_name': '华为Mate 40 pro'}, {'product_name': '华为HUAWEI二手笔记本MateBook13触屏2K全面屏'}, {'product_name': 'HAWEIAI Book 2025英特尔14 Pro满血独显笔记本'}]

> Finished chain.
